稀疏检索(sparse_vectorstore)、集成召回(稠密+稀疏检索)
```
pip install rank_bm25
```

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.vectorstores import FAISS

from langchain_openai import OpenAIEmbeddings

import sys
import os
sys.path.append("../")
from rag.rag_lecture_materials.config.config import RagConfig

from langchain_classic.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import Chroma

In [ ]:
# 向量模型
embeddings = OpenAIEmbeddings(model="Pro/BAAI/bge-m3", base_url="https://api.siliconflow.cn/v1", api_key=RagConfig.api_key)
print(embeddings)

client=<openai.resources.embeddings.Embeddings object at 0x74b380058d70> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x74b380059d30> model='Pro/BAAI/bge-m3' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base='https://api.siliconflow.cn/v1' openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True


In [4]:
loader = PyPDFLoader("./西游记影评.pdf")
# 加载分割文档
documents = loader.load_and_split()
text_splitter = RecursiveCharacterTextSplitter(separators=["。"], chunk_size=128, chunk_overlap=10)
texts_chunks = text_splitter.split_documents(documents)

构建bm25向量化的文档list

In [5]:
doc_list_1=[k.page_content for k in texts_chunks]
doc_list_1

['在众多影视作品中，《西游记》以其独特魅力脱颖而出，成为难以逾越的经典。\n这部改编自明代吴承恩同名巨著的作品，历经岁月洗礼，愈发熠熠生辉，蕴含的\n丰富文化内涵和深刻哲理，深深影响着一代又一代观众',
 '。\n《西游记》的剧情围绕唐僧师徒四人西天取经展开，他们一路上降妖除魔、历经\n九九八十一难，最终取得真经、修成正果。看似简单的故事框架，实则包含无数\n精彩情节与深刻寓意',
 '。从石猴出世、大闹天宫的叛逆反抗，到踏上取经之路后历\n经磨难的成长蜕变，剧情跌宕起伏、扣人心弦。像三打白骨精、女儿国奇遇、真\n假美猴王等经典情节，不仅展现了师徒四人面临的艰难险阻，更通过对人性、道\n德、信仰的考验，引发观众的深入思考',
 '。比如，白骨精三次变幻人形，迷惑唐僧\n师徒，孙悟空火眼金睛识破并将其打死，却因唐僧误解而被逐出师门。这一情节\n深刻揭示了人性的复杂与善恶难辨，也凸显了坚持真理的艰难。\n剧中人物形象鲜明饱满，令人印象深刻',
 '。孙悟空是核心人物，他机智勇敢、神通\n广大，拥有七十二变和火眼金睛，一个筋斗云便能翻出十万八千里。他的金箍棒\n威力无穷，打得妖魔鬼怪闻风丧胆。同时，他又具有强烈的叛逆精神，敢于挑战\n天庭权威，喊出 “皇帝轮流做，明年到我家” 的豪言壮语',
 '。但在取经过程中，他\n逐渐学会克制与担当，从最初的顽劣猴王成长为守护正义的斗战胜佛，其成长历\n程激励着无数观众勇敢追求梦想、战胜困难。唐僧慈悲善良、信念坚定，一心向\n佛，立志取得真经普度众生。他虽手无缚鸡之力，却以高尚的品德和坚定的信仰\n引领徒弟们前行',
 '。但他过于善良，常被妖怪的表象迷惑，固执己见，几次错怪孙\n悟空，这也使他的形象更加真实立体。猪八戒贪吃好色、偷懒耍滑，常因一己私\n欲闹出笑话，如在高老庄贪恋女色，取经路上多次嚷嚷着要回高老庄',
 '。但他性格\n憨厚，关键时刻也能挺身而出，对师父忠诚，为团队增添不少欢乐与温情。沙僧\n则忠厚老实、任劳任怨，默默承担着挑担等苦活累活，他话语不多，却始终坚守\n岗位，是团队中不可或缺的稳定力量',
 '。师徒四人性格迥异，却相互配合、互补长\n短，共同构成一个紧密团结的团队，完美诠释了团队合作的重要性。\n《西游记》蕴含的文化内涵博大精深。它融合了佛、道、儒三家思想，展现出中\n国传统文化的独特魅力',
 '。

In [6]:
import jieba
#指定分词器
def chinese_tokenizer(text: str):
    tokens = jieba.lcut(text)
    return [token for token in tokens] # if token not in stopwords.words('chinese')
#---------- (5) 构建bm25 retriever器 -------------
bm25_retriever = BM25Retriever.from_texts( #Create a BM25Retriever from a list of texts.
    doc_list_1, metadatas=[{"source": 1}] * len(doc_list_1),
    preprocess_func=chinese_tokenizer,
)
bm25_retriever.k = 2 #设置检索条目数=2

Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.274 seconds.
Prefix dict has been built successfully.


In [7]:
query = "西游记"
bm25_result=bm25_retriever.invoke("query")
bm25_result

[Document(metadata={'source': 1}, page_content='。\n在未来，相信《西游记》将继续陪伴着人们，其蕴含的精神财富也将永远传承下\n去 。你对《西游记》哪个情节或人物印象最为深刻呢？'),
 Document(metadata={'source': 1}, page_content='。\n《西游记》是一部具有非凡魅力的影视作品，其精彩的剧情、鲜活的人物、深厚\n的文化内涵、卓越的艺术价值和广泛的影响力，共同铸就了它的经典地位。它就\n像一座取之不尽的宝藏，值得我们反复品味、深入挖掘，从中汲取智慧与力量')]

In [ ]:
#------------构建稠密向量检索器vectorstore
vectorstore = Chroma.from_documents(texts_chunks,embeddings,collection_name="novel")
# 然后你就可以像平常一样使用vectorstore了
query="成为寒暑假的 “常客”"
retriever2 = vectorstore.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.1}
)
print(vectorstore.similarity_search(query,k=2))

[Document(metadata={'comments': '', 'sourcemodified': "D:20250519203610+08'00'", 'total_pages': 2, 'subject': '', 'creator': 'WPS 文字', 'page_label': '2', 'title': '', 'source': './西游记影评.pdf', 'trapped': '/False', 'page': 1, 'creationdate': '2025-05-19T20:36:10+08:00', 'company': '', 'producer': '', 'moddate': '2025-05-19T20:36:10+08:00', 'author': '航', 'keywords': ''}, page_content='。自播出以来，它陪伴了一代又一代观众的成长，\n成为寒暑假的 “常客”，无论何时重温，都能带来新的感动与启发。它不仅在中\n国家喻户晓，还走向世界，让全球观众领略到中国文化的魅力'), Document(metadata={'company': '', 'sourcemodified': "D:20250519203610+08'00'", 'trapped': '/False', 'total_pages': 2, 'moddate': '2025-05-19T20:36:10+08:00', 'title': '', 'producer': '', 'page': 1, 'source': './西游记影评.pdf', 'subject': '', 'comments': '', 'keywords': '', 'page_label': '2', 'creationdate': '2025-05-19T20:36:10+08:00', 'author': '航', 'creator': 'WPS 文字'}, page_content='细致入微，如各种妖魔鬼怪的形象设定、法宝的神奇功能，以及不同地域的风土\n人情，都让观众感受到中国传统文化的丰富多彩。\n这部作品的艺术价值极高。从视觉效果看，尽管早期技术有限，但剧组通过巧妙\n的场景搭建、道具制作和特效运用，打造出了一个奇幻绚丽的神话世界')]


In [9]:
#--------(6) 构建混合检索器：稠密检索(向量检索)+稀疏检索(bm25)都参与检索 并按照一定比例权重混合排序---------
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever2], weights=[0.5, 0.5]
)
#调用混合检索器进行检索,注意这里用invoke
ensemble_result=ensemble_retriever.invoke(query,k=5)
ensemble_result

[Document(metadata={'source': 1}, page_content='。自播出以来，它陪伴了一代又一代观众的成长，\n成为寒暑假的 “常客”，无论何时重温，都能带来新的感动与启发。它不仅在中\n国家喻户晓，还走向世界，让全球观众领略到中国文化的魅力'),
 Document(metadata={'source': 1}, page_content='。以《西游记》为\n蓝本的各类衍生作品层出不穷，如电影、动画、游戏、舞台剧等，不断丰富着 “西\n游” 文化的内涵，使其在新时代焕发出新的活力'),
 Document(metadata={'creator': 'WPS 文字', 'trapped': '/False', 'comments': '', 'company': '', 'page': 1, 'title': '', 'subject': '', 'creationdate': '2025-05-19T20:36:10+08:00', 'moddate': '2025-05-19T20:36:10+08:00', 'keywords': '', 'producer': '', 'source': './西游记影评.pdf', 'page_label': '2', 'total_pages': 2, 'sourcemodified': "D:20250519203610+08'00'", 'author': '航'}, page_content='细致入微，如各种妖魔鬼怪的形象设定、法宝的神奇功能，以及不同地域的风土\n人情，都让观众感受到中国传统文化的丰富多彩。\n这部作品的艺术价值极高。从视觉效果看，尽管早期技术有限，但剧组通过巧妙\n的场景搭建、道具制作和特效运用，打造出了一个奇幻绚丽的神话世界'),
 Document(metadata={'sourcemodified': "D:20250519203610+08'00'", 'title': '', 'total_pages': 2, 'keywords': '', 'creationdate': '2025-05-19T20:36:10+08:00', 'moddate': '2025-05-19T20:36:10+08:00', 'comments': '', 'tra